# Ube Craze Sentiment Analysis

This notebook runs the CardiffNLP Multilingual XLM-RoBERTa sentiment model locally using PyTorch and Hugging Face `transformers` on the preprocessed comments to identify negative, neutral, and positive attitudes towards the Ube trend.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import torch

from ube_craze_nlp.nlp.sentiment import analyze_sentiment_batch
from ube_craze_nlp.utils.paths import FINAL_DATA_DIR, PROCESSED_DATA_DIR, ensure_dirs

ensure_dirs()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Preprocessed Data

In [ ]:
processed_file = PROCESSED_DATA_DIR / "cleaned_comments.csv"

if not processed_file.exists():
    print("❌ Cleaned comments dataset not found! Please run Notebook 02 first.")
    df_comments = pd.DataFrame()
else:
    df_comments = pd.read_csv(
        processed_file, dtype={"comment_id": str, "reply_to_comment_id": str, "video_id": str}
    )
    df_comments["reply_to_comment_id"] = df_comments["reply_to_comment_id"].fillna("").astype(str)
    print(f"Loaded {len(df_comments)} cleaned comments.")
    display(df_comments.head())

## 3. Execute Sentiment Analysis

We score the comments in batches to optimize memory and speed. This will download the CardiffNLP model weights (~1.1GB) during the first run and cache them locally.

In [ ]:
if df_comments.empty:
    print("No comments to process.")
else:
    print("⏳ Running local XLM-RoBERTa sentiment inference (in batches)...")
    texts = df_comments["normalized_text"].tolist()

    # Batch execution
    predictions = analyze_sentiment_batch(texts)

    # Map outputs into dataframe columns
    df_comments["sentiment"] = [p["label"] for p in predictions]
    df_comments["sentiment_score"] = [p["score"] for p in predictions]

    print("\nSentiment analysis complete!")
    print(df_comments["sentiment"].value_counts())
    display(df_comments.head())

## 4. Export final scored comments

In [ ]:
if not df_comments.empty:
    final_file = FINAL_DATA_DIR / "sentiment_scored_comments.csv"
    df_comments["comment_id"] = df_comments["comment_id"].astype(str)
    df_comments["reply_to_comment_id"] = df_comments["reply_to_comment_id"].fillna("").astype(str)
    df_comments.to_csv(final_file, index=False, encoding="utf-8-sig")
    print(f"✅ Sentiment analysis data exported successfully to: {final_file}")